# CNN + GloVe (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_cnn` (word indices, train-only vocab).

**Run modes** (set flags in the next code cell): **quick** smoke test (`QUICK_ITERATION = True`), **progress report** (`QUICK_ITERATION = False`, `PROGRESS_REPORT_MODE = True`), or **full** (both `False`).

**Next steps for your proposal:** Download GloVe (e.g. `glove.6B.100d.txt`), implement `build_glove_weight_matrix(vocab, path, dim)` aligned to `data.vocab`, pass `embedding.weight.data.copy_(...)`. Until then this notebook uses **trainable** random embeddings so the pipeline runs.

**Metrics (course proposal):** micro/macro F1, precision/recall/ROC-AUC per label, confusion matrices, training time, parameter count.

In [1]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_cnn

# ========== Run configuration (edit here) ==========
# Three modes (quick wins if True):
#   1) quick smoke     — QUICK_ITERATION = True
#   2) progress report — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = True
#   3) full              — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = False
QUICK_ITERATION = False
PROGRESS_REPORT_MODE = True

LR = 1e-3

# --- Quick smoke-test (used only if QUICK_ITERATION) ---
QUICK_MAX_TRAIN_SAMPLES = 2048
QUICK_MAX_VAL_SAMPLES = 512
QUICK_MAX_LEN = 64
QUICK_BATCH_SIZE = 128
QUICK_EPOCHS = 1
QUICK_MAX_VOCAB = 3000
QUICK_MIN_FREQ = 1

# --- Progress-report run (used if not quick and PROGRESS_REPORT_MODE) ---
PROGRESS_MAX_TRAIN_SAMPLES = 10_000
PROGRESS_MAX_VAL_SAMPLES = 2_000
PROGRESS_MAX_LEN = 128
PROGRESS_BATCH_SIZE = 64
PROGRESS_EPOCHS = 2
PROGRESS_MAX_VOCAB = 10_000
PROGRESS_MIN_FREQ = 2

# --- Full run (both flags False) ---
FULL_MAX_TRAIN_SAMPLES = None
FULL_MAX_VAL_SAMPLES = None
FULL_MAX_LEN = 256
FULL_BATCH_SIZE = 64
FULL_EPOCHS = 1
FULL_MAX_VOCAB = 50_000
FULL_MIN_FREQ = 2

if QUICK_ITERATION:
    run_mode = "quick (smoke test)"
    _train_n = QUICK_MAX_TRAIN_SAMPLES
    _val_n = QUICK_MAX_VAL_SAMPLES
    MAX_LEN = QUICK_MAX_LEN
    BATCH_SIZE = QUICK_BATCH_SIZE
    EPOCHS = QUICK_EPOCHS
    MAX_VOCAB = QUICK_MAX_VOCAB
    MIN_FREQ = QUICK_MIN_FREQ
elif PROGRESS_REPORT_MODE:
    run_mode = "progress report"
    _train_n = PROGRESS_MAX_TRAIN_SAMPLES
    _val_n = PROGRESS_MAX_VAL_SAMPLES
    MAX_LEN = PROGRESS_MAX_LEN
    BATCH_SIZE = PROGRESS_BATCH_SIZE
    EPOCHS = PROGRESS_EPOCHS
    MAX_VOCAB = PROGRESS_MAX_VOCAB
    MIN_FREQ = PROGRESS_MIN_FREQ
else:
    run_mode = "full"
    _train_n = FULL_MAX_TRAIN_SAMPLES
    _val_n = FULL_MAX_VAL_SAMPLES
    MAX_LEN = FULL_MAX_LEN
    BATCH_SIZE = FULL_BATCH_SIZE
    EPOCHS = FULL_EPOCHS
    MAX_VOCAB = FULL_MAX_VOCAB
    MIN_FREQ = FULL_MIN_FREQ

data = preprocess_for_cnn(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)
vocab_size = len(data.vocab)
embed_dim = 100
num_labels = len(LABEL_COLUMNS)

n_train, n_val = data.X_train.shape[0], data.X_val.shape[0]
print("=" * 60)
print("CONFIG")
print("  mode:", run_mode)
print("  device:", DEVICE)
print("  train_samples:", n_train, "| val_samples:", n_val)
print("  MAX_LEN:", MAX_LEN, "| vocab_size:", vocab_size)
print("  BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS)
print("=" * 60)

CONFIG
  mode: progress report
  device: cpu
  train_samples: 10000 | val_samples: 2000
  MAX_LEN: 128 | vocab_size: 10000
  BATCH_SIZE: 64 | EPOCHS: 2


In [3]:
class TextCNN(nn.Module):
    # Kim-style multi-channel CNN over embeddings

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        num_labels: int,
        padding_idx: int = 0,
        kernel_sizes: tuple[int, ...] = (3, 4, 5),
        num_filters: int = 100,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_labels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e = self.embedding(x).transpose(1, 2)
        pools = []
        for conv in self.convs:
            h = torch.relu(conv(e))
            pools.append(h.max(dim=2).values)
        h = torch.cat(pools, dim=1)
        return self.fc(self.dropout(h))


model = TextCNN(vocab_size, embed_dim, num_labels, padding_idx=0).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_ds = TensorDataset(
    torch.tensor(data.X_train, dtype=torch.long),
    torch.tensor(data.y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(
    torch.tensor(data.X_val, dtype=torch.long),
    torch.tensor(data.y_val, dtype=torch.float32),
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

t0 = time.perf_counter()
model.train()
final_train_loss = 0.0
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    final_train_loss = epoch_loss / max(n_batches, 1)
train_seconds = time.perf_counter() - t0

model.eval()
val_loss_sum = 0.0
val_batches = 0
with torch.no_grad():
    for xb, yb in val_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        val_loss_sum += loss_fn(model(xb), yb).item()
        val_batches += 1
val_loss = val_loss_sum / max(val_batches, 1)

print(f"Training wall time ({EPOCHS} epoch(s)): {train_seconds:.1f} s")
print(f"Final train loss (last epoch avg): {final_train_loss:.4f} | val loss: {val_loss:.4f}")

Training wall time (2 epoch(s)): 25.6 s
Final train loss (last epoch avg): 0.1121 | val loss: 0.1080


In [4]:
def predict_logits(model, X_np: np.ndarray, batch_size: int = 512) -> np.ndarray:
    model.eval()
    out = []
    x = torch.tensor(X_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            logits = model(x[i : i + batch_size].to(DEVICE))
            out.append(logits.cpu().numpy())
    return np.concatenate(out, axis=0)


logits_val = predict_logits(model, data.X_val)
prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()
pred_val = (prob_val >= 0.5).astype(np.int64)

per_label_df, summary = multilabel_evaluation_report(
    data.y_val, pred_val, prob_val, LABEL_COLUMNS
)
print("Micro / macro F1:", summary)
display(per_label_df)
confs = per_label_confusion_matrices(data.y_val, pred_val, LABEL_COLUMNS)
for name, m in confs.items():
    print(name, "\n", m)

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary["f1_macro"] == 0.0 and summary["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common with very few epochs)."
        " ROC-AUC may still be > 0.5. Use progress/full mode and more EPOCHS for meaningful F1."
    )

print()
print("--- Progress report (concise) ---")
print(f"  mode: {run_mode}")
print(f"  final_train_loss: {final_train_loss:.4f}")
print(f"  val_loss: {val_loss:.4f}")
print(f"  f1_micro: {summary['f1_micro']:.4f}")
print(f"  f1_macro: {summary['f1_macro']:.4f}")

Micro / macro F1: {'f1_micro': 0.3079470198675497, 'f1_macro': 0.15303599699077416}


,label,precision,recall,f1,roc_auc
0,toxic,0.674419,0.285714,0.401384,0.858662
1,severe_toxic,0.000000,0.000000,0.000000,0.900658
2,obscene,0.739130,0.149123,0.248175,0.871116
3,threat,0.000000,0.000000,0.000000,0.579785
4,insult,0.720000,0.165138,0.268657,0.896608
5,identity_hate,0.000000,0.000000,0.000000,0.786499


toxic 
 [[1769   28]
 [ 145   58]]
severe_toxic 
 [[1975    0]
 [  25    0]]
obscene 
 [[1880    6]
 [  97   17]]
threat 
 [[1996    0]
 [   4    0]]
insult 
 [[1884    7]
 [  91   18]]
identity_hate 
 [[1985    0]
 [  15    0]]

--- Proposal summary (record for report / comparison) ---
  device: cpu
  training_time_s: 25.59
  num_parameters: 1122106

--- Progress report (concise) ---
  mode: progress report
  final_train_loss: 0.1121
  val_loss: 0.1080
  f1_micro: 0.3079
  f1_macro: 0.1530


## Proposal checklist (report)

- Compare **training time** and **parameter count** across the four model notebooks.
- **Per-label threshold tuning** on validation (default here: 0.5) for rare classes such as `threat`.
- **Model size on disk**: save `state_dict` or full checkpoint and report file size in MB.
- Optional: **macro** vs **micro** trade-offs given imbalance (already reported above).